# Module 47: DistributedDataParallel Patterns

DDP is PyTorch's standard API for multi-GPU training. This notebook covers
the key concepts without requiring an actual multi-GPU setup.

**Topics:**
- Process group setup
- DDP wrapper and gradient sync
- Gradient accumulation with `no_sync()`
- DistributedSampler for data splitting

In [ ]:
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data.distributed import DistributedSampler

# DDP wraps a model and synchronizes gradients via all-reduce
# In a real setup, each process manages one GPU:
#   torchrun --nproc_per_node=4 train.py

model = nn.Sequential(
    nn.Linear(784, 256), nn.ReLU(),
    nn.Linear(256, 10),
)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# DistributedSampler ensures each rank gets a unique data subset
dataset = TensorDataset(torch.randn(1000, 784), torch.randint(0, 10, (1000,)))

# Simulating rank 0 of 4 processes
sampler = DistributedSampler(dataset, num_replicas=4, rank=0, shuffle=True)
loader = DataLoader(dataset, batch_size=32, sampler=sampler)

print(f"Dataset size: {len(dataset)}")
print(f"Samples per rank: {len(sampler)}")
print(f"Batches per rank: {len(loader)}")

In [ ]:
# DDP training template (runs only in distributed context)
TEMPLATE = '''
import os, torch, torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

dist.init_process_group(backend="nccl")
local_rank = int(os.environ["LOCAL_RANK"])
torch.cuda.set_device(local_rank)

model = MyModel().cuda(local_rank)
ddp_model = DDP(model, device_ids=[local_rank])

for epoch in range(num_epochs):
    sampler.set_epoch(epoch)  # re-shuffle per epoch
    for x, y in loader:
        loss = loss_fn(ddp_model(x.cuda()), y.cuda())
        loss.backward()     # triggers all-reduce
        optimizer.step()
        optimizer.zero_grad()

dist.destroy_process_group()
'''
print(TEMPLATE)

In [ ]:
# Gradient accumulation pattern
# In DDP, use model.no_sync() to skip all-reduce on intermediate micro-batches

ACCUM_TEMPLATE = '''
accumulation_steps = 4
for step, (x, y) in enumerate(loader):
    is_sync_step = (step + 1) % accumulation_steps == 0
    ctx = torch.enable_grad if is_sync_step else ddp_model.no_sync
    with ctx():
        loss = loss_fn(ddp_model(x), y) / accumulation_steps
        loss.backward()
    if is_sync_step:
        optimizer.step()
        optimizer.zero_grad()
'''
print(ACCUM_TEMPLATE)

## Key Takeaways

- DDP uses **one process per GPU** — avoids the GIL bottleneck of DataParallel
- Always call `sampler.set_epoch(epoch)` to get proper shuffling
- Save checkpoints from rank 0 only: `if dist.get_rank() == 0: torch.save(...)`
- Use `model.module.state_dict()` to strip DDP wrapper keys from checkpoints
- See `ddp_training.py` for the complete training implementation